# 12 - Git & Version Control

Concise, interview/revision focused. Part of Python & DSA. Every command below runs for real in a scratch repo, including a genuine merge conflict that gets created and resolved live.

In [1]:
import subprocess, pathlib, shutil

REPO = pathlib.Path("/tmp/git_demo")
shutil.rmtree(REPO, ignore_errors=True)
REPO.mkdir()

def git(*args, cwd=REPO, check=True):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout.strip()

git("init", "-q")
git("config", "user.email", "demo@example.com")
git("config", "user.name", "Demo User")
git("branch", "-m", "main")   # normalize default branch name across git versions
print("repo initialized at", REPO)

repo initialized at /tmp/git_demo


# Part 1 - The Core Loop: status, add, commit, log

In [2]:
(REPO / "app.py").write_text("print('v1')\n")
print("status before add:\n", git("status", "-s"))

git("add", "app.py")
print("status after add (staged):\n", git("status", "-s"))

git("commit", "-q", "-m", "initial commit")
print(git("log", "--oneline"))

status before add:
 ?? app.py
status after add (staged):
 A  app.py
ce7e07b initial commit


### The three states: working directory -> staging area (index) -> repository

`add` moves changes into the STAGING area, not the repo -- `commit` is what actually records a snapshot. This two-step process is why you can stage only some of your changes (`git add -p`) while leaving others uncommitted.

In [3]:
(REPO / "app.py").write_text("print('v1')\nprint('v2 line')\n")
print("working tree modified, not staged:\n", git("status", "-s"))
print("diff (working tree vs staged):\n", git("diff"))

git("add", "-A")
print("diff --staged (staged vs last commit):\n", git("diff", "--staged"))
git("commit", "-q", "-m", "add v2 line")

working tree modified, not staged:
 M app.py
diff (working tree vs staged):
 diff --git a/app.py b/app.py
index e59f059..8ce519c 100644
--- a/app.py
+++ b/app.py
@@ -1 +1,2 @@
 print('v1')
+print('v2 line')
diff --staged (staged vs last commit):
 diff --git a/app.py b/app.py
index e59f059..8ce519c 100644
--- a/app.py
+++ b/app.py
@@ -1 +1,2 @@
 print('v1')
+print('v2 line')


''

# Part 2 - Branching & Merging (Including a Real Conflict)

In [4]:
git("checkout", "-q", "-b", "feature/greeting")
(REPO / "app.py").write_text("print('v1')\nprint('hello from feature branch')\n")
git("commit", "-q", "-am", "change greeting on feature branch")

git("checkout", "-q", "main")
(REPO / "app.py").write_text("print('v1')\nprint('hello from main')\n")
git("commit", "-q", "-am", "change greeting on main")

print("branches:\n", git("branch"))
print("\nboth branches touched the SAME line differently -- merging will conflict")
merge_result = subprocess.run(["git", "merge", "feature/greeting"], cwd=REPO, capture_output=True, text=True)
print("\nmerge exit code:", merge_result.returncode, "(non-zero = conflict)")
print(merge_result.stdout)

branches:
 feature/greeting
* main

both branches touched the SAME line differently -- merging will conflict

merge exit code: 1 (non-zero = conflict)
Auto-merging app.py
CONFLICT (content): Merge conflict in app.py
Automatic merge failed; fix conflicts and then commit the result.



In [5]:
conflicted = (REPO / "app.py").read_text()
print("conflict markers in the file:\n")
print(conflicted)

# Resolve by hand -- exactly what an editor / merge tool does
(REPO / "app.py").write_text("print('v1')\nprint('hello from main, merged with feature')\n")
git("add", "app.py")
git("commit", "-q", "-m", "merge feature/greeting, resolve conflict")
print("resolved. log:\n", git("log", "--oneline", "--graph", "--all"))

conflict markers in the file:

print('v1')
<<<<<<< HEAD
print('hello from main')
print('hello from feature branch')
>>>>>>> feature/greeting

resolved. log:
 *   087e38d merge feature/greeting, resolve conflict
|\  
| * f9f18a7 change greeting on feature branch
* | 70cf43d change greeting on main
|/  
* af18bb6 add v2 line
* ce7e07b initial commit


### merge vs rebase

`merge` creates a new commit joining both histories, preserving exactly what happened (including the conflict resolution above) -- safe for shared/public branches. `rebase` replays your commits on top of another branch, producing a linear history as if you had branched off later -- cleaner history, but REWRITES commit hashes, so rebasing a branch others already pulled causes their history to diverge from yours. Rule of thumb: rebase local/unpushed work to clean it up before sharing; merge once it is shared.

# Part 3 - Inspecting & Undoing History

In [6]:
(REPO / "broken.py").write_text("this breaks the build\n")
git("add", "broken.py")
git("commit", "-q", "-m", "oops, bad commit")
print(git("log", "--oneline", "-3"))

ce360c2 oops, bad commit
087e38d merge feature/greeting, resolve conflict
70cf43d change greeting on main


### revert vs reset -- the most-asked git interview question

`revert` creates a NEW commit that undoes an earlier one -- history is preserved, safe on shared branches, and correct for undoing something already pushed. `reset` moves the branch pointer backward, rewriting history -- `--soft` keeps changes staged, `--mixed` (default) unstages them, `--hard` discards them entirely. Never `reset` a commit others have already pulled.

In [7]:
git("revert", "--no-edit", "HEAD")
print("after revert (new commit undoing the bad one, history intact):\n", git("log", "--oneline", "-3"))

git("reset", "--hard", "HEAD~1")
print("\nafter reset --hard HEAD~1 (the revert commit itself is now GONE):\n", git("log", "--oneline", "-3"))

after revert (new commit undoing the bad one, history intact):
 5d241d1 Revert "oops, bad commit"
ce360c2 oops, bad commit
087e38d merge feature/greeting, resolve conflict

after reset --hard HEAD~1 (the revert commit itself is now GONE):
 ce360c2 oops, bad commit
087e38d merge feature/greeting, resolve conflict
70cf43d change greeting on main


### cherry-pick: taking one specific commit from another branch

Applies a single commit's changes onto the current branch without merging everything else from its source branch -- common for backporting one fix to a release branch.

In [8]:
git("checkout", "-q", "-b", "hotfix-source")
(REPO / "fix.py").write_text("critical fix\n")
git("add", "fix.py")
git("commit", "-q", "-m", "critical fix")
fix_commit = git("rev-parse", "HEAD")

git("checkout", "-q", "main")
git("cherry-pick", fix_commit)
print("fix.py now on main too:", (REPO / "fix.py").exists())
print(git("log", "--oneline", "-2"))

fix.py now on main too: True
7f6989e critical fix
ce360c2 oops, bad commit


# Part 4 - Remotes, .gitignore, stash

### A real remote, locally: push, pull, clone

No internet needed -- a bare repo on disk behaves exactly like GitHub/GitLab for push/pull/clone mechanics.

In [9]:
BARE = pathlib.Path("/tmp/git_demo_bare.git")
shutil.rmtree(BARE, ignore_errors=True)
git("init", "-q", "--bare", "-b", "main", str(BARE), cwd=REPO)

git("remote", "add", "origin", str(BARE))
git("push", "-q", "-u", "origin", "main")
print("pushed. remote branches visible from the bare repo:")
print(subprocess.run(["git", "branch"], cwd=BARE, capture_output=True, text=True).stdout)

CLONE = pathlib.Path("/tmp/git_demo_clone")
shutil.rmtree(CLONE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", str(BARE), str(CLONE)], capture_output=True, text=True)
print("cloned files:", sorted(p.name for p in CLONE.iterdir() if not p.name.startswith(".git")))

pushed. remote branches visible from the bare repo:
* main



cloned files: ['app.py', 'broken.py', 'fix.py']


### .gitignore and git stash

In [10]:
(REPO / ".gitignore").write_text("*.log\n__pycache__/\n.venv/\n")
(REPO / "debug.log").write_text("noisy output\n")
git("add", "-A")
print("status: debug.log is NOT staged, .gitignore is:\n", git("status", "-s"))
git("commit", "-q", "-m", "add gitignore")

(REPO / "app.py").write_text((REPO / "app.py").read_text() + "print('half-finished change')\n")
git("stash")
print("after stash, working tree is clean:\n", git("status", "-s"))
git("stash", "pop")
print("after pop, change is back:\n", git("status", "-s"))

status: debug.log is NOT staged, .gitignore is:
 A  .gitignore


after stash, working tree is clean:
 


after pop, change is back:
 M app.py


## Interview rapid-fire

- `git pull` = `git fetch` + `git merge` in one step -- `fetch` alone updates your knowledge of the remote without touching your working files, which is why `fetch` is the safer default when you just want to LOOK at what changed.
- Detached HEAD: checking out a specific commit (not a branch) puts you here -- commits made in this state are not on any branch and can be garbage-collected/lost unless you create a branch from them before switching away.
- `HEAD` is a pointer to "whatever commit you currently have checked out," not a fixed commit -- it moves with every commit/checkout.
- A `.git` directory holds the ENTIRE history locally -- this is why git is called distributed: every clone is a full backup, not just a working copy.
- Squash merge: combines an entire feature branch into ONE commit on main -- clean history, but loses the individual commit-by-commit story (a real tradeoff, not strictly better than a regular merge).

## Practice

1. Create two branches that both modify the same line of the same file, merge them, and resolve the conflict -- reproduce Part 2 from scratch without copying the code above.
2. Make 3 commits, then use `git reset --soft HEAD~2` and inspect `git status` -- explain what state the changes end up in.
3. Push to the bare remote from Part 4, clone it a second time, make a commit in the second clone, push, then `git pull` in the first clone and confirm the new commit appears.